# Lucid Classifier Inspection

This notebook inspects the Lucid-style classifier used to predict sharing-score labels from solo-profile features.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

FEATURES_LUCID_FAITHFUL = [
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "amp_enabled",
]

FEATURES_EXTENDED = [
    *FEATURES_LUCID_FAITHFUL,
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "horus_gpu_util_p95",
    "horus_gpu_util_max",
]

feature_table_path = Path("../results/lucid_feature_table.csv")
df = pd.read_csv(feature_table_path)
df.head()


,spec_name,spec_path,spec_key,peak_memory_mib,memory_fraction,horus_gpu_util_mean,horus_gpu_util_p95,horus_gpu_util_max,avg_smact,avg_smocc,...,representative_spec_path,lucid_mean_normalized_speed,lucid_std_normalized_speed,lucid_min_normalized_speed,lucid_max_normalized_speed,lucid_num_pair_observations,lucid_class,lucid_ss,lucid_label_source,lucid_label_usable
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_cifar100_bs128_20e_1gpu.yaml,860,0.020996,37.251852,44.0,56.0,0.226215,0.143911,...,evaluation/workloads/training/specs/yaml_thres...,0.950506,0.116516,0.574695,1.037478,14,tiny,0,measured_pairwise,True
1,efficientnet_imagenet_bs128_maxbatches1200.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_imagenet_bs128_1gpu.yaml,12732,0.310840,75.634328,85.0,86.0,0.605575,0.374403,...,evaluation/workloads/training/specs/yaml_thres...,0.833739,0.156536,0.490306,0.981883,13,jumbo,2,measured_pairwise,True
2,efficientnet_imagenet_bs64_maxbatches1200.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_imagenet_bs64_1gpu.yaml,6724,0.164160,77.585185,82.0,83.0,0.602948,0.363704,...,evaluation/workloads/training/specs/yaml_thres...,0.807846,0.157202,0.403917,0.983718,21,jumbo,2,measured_pairwise,True
3,llama3_width_8layer_wiki_bs1_1gpu_maxsteps2000...,evaluation/workloads/training/specs/yaml_thres...,llama3_width_8layer_wiki_bs1_1gpu.yaml,28712,0.700977,97.698795,99.0,99.0,0.295530,0.166651,...,evaluation/workloads/training/specs/yaml_thres...,0.895632,0.104657,0.656576,0.979835,9,medium,1,measured_pairwise,True
4,mobilenet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml_thres...,mobilenet_cifar100_bs128_20e_1gpu.yaml,634,0.015479,32.615385,40.0,46.0,0.157754,0.090031,...,evaluation/workloads/training/specs/yaml_thres...,0.898231,0.161183,0.486047,1.061015,15,medium,1,measured_pairwise,True


## Label coverage

In [2]:
print("Rows:", len(df))
print("\nUsable label counts:")
print(df["lucid_label_usable"].value_counts(dropna=False))

print("\nClass counts:")
print(df["lucid_class"].value_counts(dropna=False))

df[[
    "spec_key",
    "lucid_num_pair_observations",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
]].sort_values(["lucid_label_usable", "lucid_num_pair_observations"], ascending=[True, True])


Rows: 20

Usable label counts:
lucid_label_usable
True    20
Name: count, dtype: int64

Class counts:
lucid_class
jumbo     10
medium     7
tiny       3
Name: count, dtype: int64


,spec_key,lucid_num_pair_observations,lucid_label_usable,lucid_class,lucid_ss
5,mobilenet_cifar100_bs64_20e_1gpu.yaml,4,True,tiny,0
10,resnet34_cifar100_bs64_20e_1gpu.yaml,4,True,tiny,0
3,llama3_width_8layer_wiki_bs1_1gpu.yaml,9,True,medium,1
17,xception_imagenet_bs128_1gpu.yaml,10,True,medium,1
8,resnet18_cifar100_bs64_20e_1gpu.yaml,11,True,medium,1
11,resnet50_imagenet_bs128_1gpu.yaml,12,True,jumbo,2
1,efficientnet_imagenet_bs128_1gpu.yaml,13,True,jumbo,2
13,unet_voc_1gpu_10e_1gpu.yaml,13,True,jumbo,2
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,14,True,tiny,0
6,mobilenet_imagenet_bs128_1gpu.yaml,14,True,medium,1


## Train/evaluate classifier

In [3]:
def train_and_report(feature_cols, seed=42):
    train = df[df["lucid_label_usable"] == True].copy()
    X = train[feature_cols]
    y = train["lucid_ss"].astype(int)

    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=seed,
                    class_weight="balanced",
                    min_samples_leaf=1,
                ),
            ),
        ]
    )

    model.fit(X, y)

    if len(train) >= 3:
        pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
        print("Leave-one-out report")
        print(classification_report(y, pred, zero_division=0))
        print("Confusion matrix [0=tiny, 1=medium, 2=jumbo]")
        print(confusion_matrix(y, pred, labels=[0, 1, 2]))

    importances = model.named_steps["clf"].feature_importances_
    imp = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp = imp.sort_values("importance", ascending=False)

    return model, imp

model_lucid, imp_lucid = train_and_report(FEATURES_LUCID_FAITHFUL)
imp_lucid


Leave-one-out report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.50      0.43      0.46         7
           2       0.73      0.80      0.76        10

    accuracy                           0.55        20
   macro avg       0.41      0.41      0.41        20
weighted avg       0.54      0.55      0.54        20

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[0 2 1]
 [2 3 2]
 [1 1 8]]


,feature,importance
1,memory_fraction,0.408228
0,peak_memory_mib,0.373531
2,horus_gpu_util_mean,0.218241
3,amp_enabled,0.000000


## Extended feature set

In [4]:
model_ext, imp_ext = train_and_report(FEATURES_EXTENDED)
imp_ext


Leave-one-out report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.60      0.43      0.50         7
           2       0.75      0.90      0.82        10

    accuracy                           0.60        20
   macro avg       0.45      0.44      0.44        20
weighted avg       0.58      0.60      0.58        20

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[0 2 1]
 [2 3 2]
 [1 0 9]]


,feature,importance
0,peak_memory_mib,0.198195
1,memory_fraction,0.196874
8,horus_gpu_util_max,0.118821
7,horus_gpu_util_p95,0.110413
6,avg_drama,0.110298
4,avg_smact,0.098423
2,horus_gpu_util_mean,0.089463
5,avg_smocc,0.077513
3,amp_enabled,0.000000


## Compare measured and predicted labels

In [5]:
def predict_table(model, feature_cols, source_name):
    out = df.copy()
    pred = model.predict(out[feature_cols])
    proba = model.predict_proba(out[feature_cols])
    classes = list(model.named_steps["clf"].classes_)

    ss_to_class = {0: "tiny", 1: "medium", 2: "jumbo"}

    out[f"pred_ss_{source_name}"] = pred
    out[f"pred_class_{source_name}"] = [ss_to_class[int(x)] for x in pred]

    for i, cls in enumerate(classes):
        out[f"pred_proba_ss{int(cls)}_{source_name}"] = proba[:, i]

    return out

pred_lucid = predict_table(model_lucid, FEATURES_LUCID_FAITHFUL, "lucid")
pred_lucid[[
    "spec_key",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
    "pred_class_lucid",
    "pred_ss_lucid",
    "pred_proba_ss0_lucid",
    "pred_proba_ss1_lucid",
    "pred_proba_ss2_lucid",
]].sort_values("spec_key")


,spec_key,lucid_label_usable,lucid_class,lucid_ss,pred_class_lucid,pred_ss_lucid,pred_proba_ss0_lucid,pred_proba_ss1_lucid,pred_proba_ss2_lucid
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,True,tiny,0,tiny,0,0.735,0.150,0.115
1,efficientnet_imagenet_bs128_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.225,0.775
2,efficientnet_imagenet_bs64_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.000,1.000
3,llama3_width_8layer_wiki_bs1_1gpu.yaml,True,medium,1,medium,1,0.000,0.950,0.050
4,mobilenet_cifar100_bs128_20e_1gpu.yaml,True,medium,1,medium,1,0.220,0.770,0.010
5,mobilenet_cifar100_bs64_20e_1gpu.yaml,True,tiny,0,tiny,0,0.645,0.345,0.010
6,mobilenet_imagenet_bs128_1gpu.yaml,True,medium,1,medium,1,0.000,0.805,0.195
7,mobilenet_imagenet_bs64_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.015,0.985
8,resnet18_cifar100_bs64_20e_1gpu.yaml,True,medium,1,medium,1,0.205,0.785,0.010
9,resnet34_cifar100_bs128_20e_1gpu.yaml,True,jumbo,2,jumbo,2,0.345,0.015,0.640


## Export notebook predictions

In [6]:
output = Path("../results/lucid_classifier_predictions_notebook.csv")
pred_lucid.to_csv(output, index=False)
print(output)


../results/lucid_classifier_predictions_notebook.csv
